In [ ]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random

shiftData = pd.read_csv('nhl_shifts.csv')
shiftData = shiftData[1:1000]

def time_to_sec(t):
    m, s = map(int, t.split(':'))
    return m * 60 + s

# print(shiftData)

playerIDs = shiftData['playerID'].unique()

team_ids = shiftData['teamId'].unique()

shifts_list = shiftData.to_dict(orient='records')
max_end_time = max(time_to_sec(shift['endTime']) for shift in shifts_list)

print(max_end_time)

# convert all period + time to seconds
def per_time_to_sec(period, time):
    if period == 1:
        return time_to_sec(time)
    elif period == 2:
        return 1200 + time_to_sec(time)
    elif period == 3:
        return 2400 + time_to_sec(time)
    elif period == 4:
        return 3600 + time_to_sec(time)
    else:
        raise ValueError(f"Invalid period: {period}")
    

print(shifts_list)

for shift in shifts_list:
    converted_start = per_time_to_sec(shift['period'], shift['startTime'])
    converted_end = per_time_to_sec(shift['period'], shift['endTime'])

players = []
for i in range(max_end_time + 1):
    second_players = []
    for playerID in playerIDs:
        if i>=converted_start and i<converted_end:
            second_players.append(playerID)
    players.append(second_players)

print(players)


    




# on_ice = [{tid: set() for tid in team_ids} for _ in range(max_end_time + 1)]


# for shift in shifts_list:
#     pid = shift['playerID']
#     team = shift['teamId']
#     start = time_to_sec(shift['startTime'])
#     end = time_to_sec(shift['endTime'])
#     for t in range(start, end):
#         if 0 <= t < len(on_ice):
#             on_ice[t][team].add(pid)





1200
[{'gameID': 2021021000, 'shiftID': 11490107, 'playerID': 8470638, 'shiftNumber': 1, 'period': 1, 'startTime': '00:00', 'endTime': '00:22', 'duration': '00:22', 'teamId': 6, 'typeCode': 517, 'detailCode': 0, 'eventDescription': nan, 'eventNumber': 6, 'hexValue': '#111111'}, {'gameID': 2021021000, 'shiftID': 11490108, 'playerID': 8470638, 'shiftNumber': 2, 'period': 1, 'startTime': '02:34', 'endTime': '03:31', 'duration': '00:57', 'teamId': 6, 'typeCode': 517, 'detailCode': 0, 'eventDescription': nan, 'eventNumber': 156, 'hexValue': '#111111'}, {'gameID': 2021021000, 'shiftID': 11490109, 'playerID': 8470638, 'shiftNumber': 3, 'period': 1, 'startTime': '06:15', 'endTime': '07:37', 'duration': '01:22', 'teamId': 6, 'typeCode': 517, 'detailCode': 0, 'eventDescription': nan, 'eventNumber': 167, 'hexValue': '#111111'}, {'gameID': 2021021000, 'shiftID': 11490110, 'playerID': 8470638, 'shiftNumber': 4, 'period': 1, 'startTime': '09:22', 'endTime': '10:32', 'duration': '01:10', 'teamId': 6,

In [5]:

# Assume: on_ice = [{team_id1: set(...), team_id2: set(...)}, ...]
# Ensure that team ids are integers and not np.int64

rows = []

# Get consistent team ordering (assuming 2 teams per game)
team_ids = list(on_ice[0].keys())  # assuming both teams always present

team1, team2 = int(team_ids[0]), int(team_ids[1])  # ensure team IDs are integers

for second, second_data in enumerate(on_ice):
    # Ensure each team is converted to an integer key
    team1_players = ','.join(map(str, second_data[int(team1)]))
    team2_players = ','.join(map(str, second_data[int(team2)]))

    row = {
        'second': second,
        f'team_{team1}': team1_players,
        f'team_{team2}': team2_players,
    }
    rows.append(row)

# Convert to DataFrame
df = pd.DataFrame(rows)

# Export to CSV
df.to_csv('on_ice_by_second.csv', index=False)


In [7]:
df

,second,team_6,team_8
0,0,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
1,1,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
2,2,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
3,3,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
4,4,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
...,...,...,...
1196,1196,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."
1197,1197,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."
1198,1198,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."
1199,1199,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."


In [12]:
df['team_6'][0]

'8475745,8478498,8476999,8473419,8470638,8475287,8475225,8476891,8479325'

In [10]:
count = df[df['team_6'].apply(lambda x: '8478498' in x)].count()
print(count)

second    804
team_6    804
team_8    804
dtype: int64
